In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import networkx as nx  # Graph handling
from torch_geometric.data import Data  # PyTorch Geometric for Graphs
from torch_geometric.loader import DataLoader
import torch_geometric.nn as pyg_nn  # Import PyG layers
from rdkit import Chem  # RDKit for molecule processing
from rdkit.Chem import AllChem
import random

In [2]:
from sklearn.preprocessing import StandardScaler

#scaler = StandardScaler()
#real_features = scaler.fit_transform(real_features)
#generated_features = scaler.transform(generated_features)

In [3]:
# Function to convert SMILES to molecular graph representation
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  # Skip invalid molecules
    
    # Extract atoms and bonds
    atom_features = torch.tensor([atom.GetAtomicNum() for atom in mol.GetAtoms()], dtype=torch.float32).unsqueeze(1)
    edge_index = []
    
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index.append((i, j))
        edge_index.append((j, i))  # Add both directions

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    return Data(x=atom_features, edge_index=edge_index)

In [4]:
# -------------------
# Define Generator (Graph-Based)
# -------------------
class GraphGenerator(nn.Module):
    def __init__(self, latent_dim, node_features=11, num_nodes=5):  # Updated for QM9
        super(GraphGenerator, self).__init__()
        self.num_nodes = num_nodes

        # Increase feature complexity to better model QM9 properties
        self.fc = nn.Linear(latent_dim, node_features * 2)  # Output twice the number of features
        self.bn = nn.BatchNorm1d(node_features * 2)  # Batch Normalization

    def forward(self, z):
        batch_size, num_nodes, _ = z.shape  # Extract correct dimensions
        node_embeddings = self.fc(z)

        # Ensure reshaping is valid
        expected_size = batch_size * num_nodes * self.fc.out_features
        if node_embeddings.numel() != expected_size:
            raise RuntimeError(
                f"Shape mismatch: expected {expected_size} elements, got {node_embeddings.numel()}. "
                f"z shape: {z.shape}, node_embeddings shape: {node_embeddings.shape}"
            )

        # Correct reshape
        node_embeddings = node_embeddings.view(batch_size, num_nodes, self.fc.out_features)

        return node_embeddings, self.create_edges(node_embeddings)

    def create_edges(self, node_embeddings, edge_prob=0.8):
        """Probabilistic edge creation ensuring bidirectionality."""
        num_nodes = node_embeddings.shape[1]  # Use correct dimension
        edges = []
        for i in range(num_nodes):
            for j in range(i + 1, num_nodes):  # Avoid self-loops & redundant edges
                if random.random() < edge_prob:
                    edges.append((i, j))
                    edges.append((j, i))  # Ensure bidirectionality

        # Handle empty edge cases
        if len(edges) == 0:
            return torch.empty((2, 0), dtype=torch.long)  

        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        return edge_index

In [5]:
# -------------------
# Define Discriminator (Graph-Based)
# -------------------
class GraphDiscriminator(nn.Module):
    def __init__(self, node_features=11):  # Updated for QM9 dataset
        super(GraphDiscriminator, self).__init__()

        # Update input channels to match QM9 (11 features per node)
        self.conv1 = pyg_nn.GCNConv(node_features, 32)  
        self.conv2 = pyg_nn.GCNConv(32, 16)

        # Fully connected layer to classify real vs. fake graphs
        self.fc = nn.Linear(16, 1)

    def forward(self, node_features, edge_index):
        """
        Parameters:
        - node_features: [num_nodes, num_features] Tensor
        - edge_index: [2, num_edges] Tensor (graph connectivity)

        Returns:
        - Validity score (scalar) indicating whether the graph is real or fake
        """
        x = torch.relu(self.conv1(node_features, edge_index))
        x = torch.relu(self.conv2(x, edge_index))

        # Aggregate all node features into a single graph representation
        graph_embedding = x.mean(dim=0)  # Global feature pooling

        return self.fc(graph_embedding)  # Output values

In [6]:
from torch.autograd import grad

def compute_gradient_penalty(discriminator, real_nodes, fake_nodes, real_edges):
    alpha = torch.rand(real_nodes.shape[0], 1, device=real_nodes.device)
    interpolates = (alpha * real_nodes + (1 - alpha) * fake_nodes).requires_grad_(True)

    d_interpolates = discriminator(interpolates, real_edges)

    gradients = grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=torch.ones_like(d_interpolates),
        create_graph=True,
        retain_graph=True,
    )[0]

    # Only take a subset of gradients to speed up training
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty * 0.5  # Reduce contribution to speed up training

In [7]:
def train_GAN(num_epochs, generator, discriminator, data_loader, latent_dim, λ=10):
    optimizer_G = optim.Adam(generator.parameters(), lr=0.0002)
    optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=0.00005)
    num_nodes = 5

    for epoch in range(num_epochs):
        for batch in data_loader:
            node_features = batch.x
            edge_index = batch.edge_index
            fuel_labels = batch.y[:, [7, 8, 9, 11, 4]]  # Extract fuel properties: [U0, U, H, Cv, Δε]

            # Generate latent vector
            z = torch.randn((batch_size, num_nodes, latent_dim))  # Ensure correct shape

            # Generate molecular graph
            generated_nodes, generated_edges = generator(z)
            
            # DETACH GENERATED DATA to avoid retaining the graph
            generated_nodes = generated_nodes.detach()
            generated_edges = generated_edges.detach()
            
            generated_nodes += torch.randn_like(generated_nodes) * 0.3  # Add small noise

            # Compute Discriminator Loss (WGAN-GP)
            real_fuel_values = discriminator(node_features, edge_index)  # Real molecules
            fake_fuel_values = discriminator(generated_nodes, generated_edges)  # Fake molecules
            
            # Compute fuel property loss (MSE)
            fuel_loss = torch.nn.MSELoss()(real_fuel_values, fuel_labels)

            # Compute gradient penalty
            gradient_penalty = compute_gradient_penalty(discriminator, node_features, generated_nodes, edge_index)
            
            loss_D = -real_fuel_values.mean() + fake_fuel_values.mean() + λ * gradient_penalty + fuel_loss  # WGAN + Fuel property loss

            optimizer_D.zero_grad()
            loss_D.backward(retain_graph=True)  # Ensure we keep the graph for Generator training
            optimizer_D.step()

            # Train Generator (Fuel-aware generation)
            fake_fuel_values = discriminator(generated_nodes, generated_edges)  # Recalculate fake score
            loss_G = -fake_fuel_values.mean()

            optimizer_G.zero_grad()
            loss_G.backward()  # No retain_graph needed for Generator
            optimizer_G.step()

        # Logging
        if epoch % 5 == 0:
            epochs.append(epoch)
            D_loss.append(loss_D.item())
            G_loss.append(loss_G.item())
            print(f"Epoch [{epoch}/{num_epochs}], Loss_D: {loss_D.item()}, Loss_G: {loss_G.item()}, Fuel_Loss: {fuel_loss.item()}")

In [8]:
from torch_geometric.datasets import QM9
dataset = QM9(root='data/QM9')
subset_size = 5000  # Only use 5,000 graphs per epoch
subset = dataset[:subset_size] 
print(dataset[0])  # Inspect QM9 properties
batch_size=64
data_loader = DataLoader(subset, batch_size, shuffle=True)

Data(x=[5, 11], edge_index=[2, 8], edge_attr=[8, 4], y=[1, 19], pos=[5, 3], z=[5], smiles='[H]C([H])([H])[H]', name='gdb_1', idx=[1])


In [9]:
# -------------------
# Run the Training with Placeholder Dataset
# -------------------
latent_dim = 32
num_epochs = 100
λ = 10  # Regularization coefficient for gradient penalty

node_features = dataset.num_node_features  # Ensure it matches QM9's 11 features

In [10]:
# Update models with correct feature size
generator = GraphGenerator(latent_dim, node_features)
discriminator = GraphDiscriminator(node_features)

epochs = []
D_loss = []
G_loss = []

train_GAN(num_epochs, generator, discriminator, data_loader, latent_dim)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (320x22 and 11x32)

In [ ]:
import matplotlib.pyplot as plt

# Plot Discriminator and Generator Loss
plt.figure(figsize=(8, 5))
plt.plot(epochs, D_loss, label="Discriminator Loss (D)", marker='o', linestyle='-')
plt.plot(epochs, G_loss, label="Generator Loss (G)", marker='s', linestyle='--')

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("GAN Training Loss Over Time")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.utils import to_networkx
from torch_geometric.data import Data

# Generate graph
z = torch.randn((5, latent_dim))  # Ensure multiple nodes
node_features, edge_index = generator(z)
graph_data = Data(x=node_features, edge_index=edge_index)  # Wrap in Data object

# Convert and plot
G = to_networkx(graph_data, to_undirected=True)
plt.figure(figsize=(5, 5))
nx.draw(G, with_labels=True, node_color="lightblue", edge_color="gray", node_size=500)
plt.title("Generated Graph from GAN")
plt.show()

In [ ]:
import seaborn as sns
import torch

# Get a batch of real data from the DataLoader
real_batch = next(iter(data_loader))  # Fetch the first batch from real dataset
real_node_features = real_batch.x  # Extract node features from real data

# Generate a batch of fake data using the trained generator
z = torch.randn((1, latent_dim))  # Random latent vector
generated_node_features, _ = generator(z)  # Get generated node features

# Convert PyTorch tensors to NumPy for plotting
real_features = real_node_features.flatten().detach().cpu().numpy()
generated_features = generated_node_features.flatten().detach().cpu().numpy()

# Plot feature distributions
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
sns.kdeplot(real_features, label="Real Features", fill=True, color="blue", alpha=0.5)
sns.kdeplot(generated_features, label="Generated Features", fill=True, color="red", alpha=0.5)

plt.xlabel("Feature Value")
plt.ylabel("Density")
plt.title("Distribution of Real vs. Generated Node Features")
plt.legend()
plt.show()
